In [1]:
from pyspark.sql.functions import col, to_date, col, lit, when, substring
from pyspark.sql.functions import lower, regexp_replace, trim
from pyspark.sql.types import StructType, StructField, StringType


StatementMeta(, 6dcd164d-fc23-45b0-b81a-95af71e120fd, 3, Finished, Available, Finished, False)

In [2]:
# Load the three source CSV files into separate Spark DataFrames.
df1 = spark.read.format("csv")\
.option("header","true")\
.load("abfss://SSR_DSML_MINI_PROJECT@onelake.dfs.fabric.microsoft.com/DSMLLH1.Lakehouse/Files/raw/api_data_aadhar_enrolment/api_data_aadhar_enrolment_0_500000.csv")

df2 = spark.read.format("csv").option("header","true").load("abfss://SSR_DSML_MINI_PROJECT@onelake.dfs.fabric.microsoft.com/DSMLLH1.Lakehouse/Files/raw/api_data_aadhar_enrolment/api_data_aadhar_enrolment_1000000_1006029.csv")

df3 = spark.read.format("csv").option("header","true").load("abfss://SSR_DSML_MINI_PROJECT@onelake.dfs.fabric.microsoft.com/DSMLLH1.Lakehouse/Files/raw/api_data_aadhar_enrolment/api_data_aadhar_enrolment_500000_1000000.csv")

# Preview the first few records to confirm that the data was loaded correctly.
display(df1.limit(5))


StatementMeta(, 6dcd164d-fc23-45b0-b81a-95af71e120fd, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a3f72fac-d22c-4556-9b66-e456e3383387)

In [3]:
enrol_df = df1.union(df2).union(df3)

# Display a sample of the combined DataFrame.
display(enrol_df.limit(5))


StatementMeta(, 6dcd164d-fc23-45b0-b81a-95af71e120fd, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 73a2b55b-9aa6-4eb5-b30b-71b8be225407)

In [4]:
print(f"df schema : {enrol_df.printSchema()}")
print(f"df describe : {enrol_df.describe()}")
print(f"df rowCount : {enrol_df.count()}")

StatementMeta(, 6dcd164d-fc23-45b0-b81a-95af71e120fd, 6, Finished, Available, Finished, False)

root
 |-- date: string (nullable = true)
 |-- state: string (nullable = true)
 |-- district: string (nullable = true)
 |-- pincode: string (nullable = true)
 |-- age_0_5: string (nullable = true)
 |-- age_5_17: string (nullable = true)
 |-- age_18_greater: string (nullable = true)

df schema : None
df describe : DataFrame[summary: string, date: string, state: string, district: string, pincode: string, age_0_5: string, age_5_17: string, age_18_greater: string]
df rowCount : 1006029


In [5]:
# Apply the required data type transformations to the enrolment dataset.
enrol_df = enrol_df.select(
    # Convert the date string into a Spark date type.
    to_date(col("date") , "dd-MM-yyyy").alias("date"),
    
    # Keep location fields as strings.
    col("state").cast("string"),
    col("district").cast("string"),
    
    # Convert pincode and age-group counts to integer values.
    col("pincode").cast("integer"),
    col("age_0_5").cast("integer"),
    col("age_5_17").cast("integer"),
    col("age_18_greater").cast("integer")
)

# Verify that the transformations produced the expected schema.
enrol_df.printSchema()


StatementMeta(, 6dcd164d-fc23-45b0-b81a-95af71e120fd, 7, Finished, Available, Finished, False)

root
 |-- date: date (nullable = true)
 |-- state: string (nullable = true)
 |-- district: string (nullable = true)
 |-- pincode: integer (nullable = true)
 |-- age_0_5: integer (nullable = true)
 |-- age_5_17: integer (nullable = true)
 |-- age_18_greater: integer (nullable = true)



In [6]:
enrol_df.select("state").distinct().show()

StatementMeta(, 6dcd164d-fc23-45b0-b81a-95af71e120fd, 8, Finished, Available, Finished, False)

+--------------------+
|               state|
+--------------------+
|            Nagaland|
|           Karnataka|
|              Odisha|
|              Kerala|
|         WEST BENGAL|
|Dadra and Nagar H...|
|              Ladakh|
|          Tamil Nadu|
|        Chhattisgarh|
|      Andhra Pradesh|
|         Lakshadweep|
|      Madhya Pradesh|
|              Punjab|
|             Manipur|
|         Daman & Diu|
|     Jammu & Kashmir|
|                 Goa|
|             Mizoram|
|Dadra and Nagar H...|
|    Himachal Pradesh|
+--------------------+
only showing top 20 rows



In [7]:
# Inspect records where the state contains the invalid value "100000".
enrol_df.filter(col("state") == "100000").show()

# Count the number of records containing the invalid state value.
count = enrol_df.filter(col("state") == "100000").count()
print(f"Total fields with incorrect data: {count}")

# Remove records containing the invalid state value.
enrol_df = enrol_df.filter(col("state") != "100000")

StatementMeta(, 6dcd164d-fc23-45b0-b81a-95af71e120fd, 9, Finished, Available, Finished, False)

+----------+------+--------+-------+-------+--------+--------------+
|      date| state|district|pincode|age_0_5|age_5_17|age_18_greater|
+----------+------+--------+-------+-------+--------+--------------+
|2025-09-02|100000|  100000| 100000|      0|       0|             3|
|2025-09-03|100000|  100000| 100000|      0|       0|             1|
|2025-09-08|100000|  100000| 100000|      0|       0|             1|
|2025-09-09|100000|  100000| 100000|      0|       0|             1|
|2025-09-11|100000|  100000| 100000|      0|       0|             2|
|2025-09-12|100000|  100000| 100000|      0|       0|             2|
|2025-09-19|100000|  100000| 100000|      0|       0|             1|
|2025-09-20|100000|  100000| 100000|      0|       0|             1|
|2025-10-24|100000|  100000| 100000|      0|       1|             0|
|2025-11-15|100000|  100000| 100000|      0|       0|             3|
|2025-11-16|100000|  100000| 100000|      0|       0|             2|
|2025-11-17|100000|  100000| 10000

In [8]:
def normalize_text(col_obj):
    # Convert text to lowercase so that comparisons are case-insensitive.
    c = lower(col_obj)

    # Standardise the ampersand character.
    c = regexp_replace(c, "&", "and")

    # Replace spaces, dots, and hyphens with underscores.
    c = regexp_replace(c, r"[\s\.\-]+", "_")

    # Collapse consecutive underscores into a single underscore.
    c = regexp_replace(c, r"_+", "_")

    # Remove "the_" when it appears at the beginning of a value.
    c = regexp_replace(c, r"^the_", "")

    # Remove unwanted underscores and asterisks from the beginning and end.
    c = regexp_replace(c, r"^[_*]+", "")
    c = regexp_replace(c, r"[_*]+$", "")
    
    return trim(c)


StatementMeta(, 6dcd164d-fc23-45b0-b81a-95af71e120fd, 10, Finished, Available, Finished, False)

In [9]:
enrol_df = enrol_df \
    .withColumnRenamed("state", "oldState") \
    .withColumnRenamed("district", "oldDistrict") \
    .withColumn("state", normalize_text(col("oldState"))) \
    .withColumn("district", normalize_text(col("oldDistrict")))

# Verify the resulting schema and inspect sample records.
enrol_df.printSchema()
display(enrol_df.limit(10))

StatementMeta(, 6dcd164d-fc23-45b0-b81a-95af71e120fd, 11, Finished, Available, Finished, False)

root
 |-- date: date (nullable = true)
 |-- oldState: string (nullable = true)
 |-- oldDistrict: string (nullable = true)
 |-- pincode: integer (nullable = true)
 |-- age_0_5: integer (nullable = true)
 |-- age_5_17: integer (nullable = true)
 |-- age_18_greater: integer (nullable = true)
 |-- state: string (nullable = true)
 |-- district: string (nullable = true)



SynapseWidget(Synapse.DataFrame, 86b0738c-c0d0-4af2-b914-b03c49dd6968)

In [10]:
states_df = spark.read.format("csv").option("header","true").load("abfss://SSR_DSML_MINI_PROJECT@onelake.dfs.fabric.microsoft.com/DSMLLH1.Lakehouse/Files/raw/STATES.csv")

# Normalise location names and convert pincode to an integer.
states_df = states_df \
    .withColumn("state", normalize_text(col("statename"))) \
    .withColumn("district", normalize_text(col("district"))) \
    .withColumn("pincode", col("pincode").cast("integer")) \
    .select("state", "district", "pincode") \
    .filter(col("state").isNotNull() & (col("state") != "na"))

# Display the distinct state names available in the reference dataset.
display(states_df.select("state").distinct())

StatementMeta(, 6dcd164d-fc23-45b0-b81a-95af71e120fd, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d4abb2f5-d87b-4b8c-86f9-8f7770f66e8d)

In [11]:
missing_states = enrol_df.join(
    states_df, 
    on="state", 
    how="left_anti"
).select("state").distinct()

display(missing_states)

StatementMeta(, 6dcd164d-fc23-45b0-b81a-95af71e120fd, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7d63a14e-12a1-4b90-83e9-3b4387e6a964)

In [12]:
from pyspark.sql import functions as F

enrol_df = enrol_df.withColumn(
    "state", 
    F.when(F.col("state") == "pondicherry", "puducherry")
     .when(F.col("state") == "orissa", "odisha")
     .when(F.col("state").isin("westbengal", "west_bangal"), "west_bengal")
     .when(F.col("state").isin("dadra_and_nagar_haveli", "daman_and_diu"), "dadra_and_nagar_haveli_and_daman_and_diu")
     .otherwise(F.col("state"))
)


StatementMeta(, 6dcd164d-fc23-45b0-b81a-95af71e120fd, 14, Finished, Available, Finished, False)

In [13]:
missing_states = enrol_df.join(
    states_df, 
    on="state", 
    how="left_anti"
).select("state").distinct()

missing_states.show()


StatementMeta(, 6dcd164d-fc23-45b0-b81a-95af71e120fd, 15, Finished, Available, Finished, False)

+-----+
|state|
+-----+
+-----+



In [14]:
# Validate the state-district combination against the trusted reference data.
missing_districts = enrol_df.join(
    states_df.select("state", "district").distinct(),
    on=["state", "district"],
    how="left_anti"
).select("state", "district").distinct()

print(f"Number of unmatched state-district combinations: {missing_districts.count()}")

StatementMeta(, 6dcd164d-fc23-45b0-b81a-95af71e120fd, 16, Finished, Available, Finished, False)

Number of unmatched state-district combinations: 254


In [15]:
import pyspark.sql.functions as F

# Identify records whose state-district combination does not exist
# in the trusted reference dataset.
invalid_location_df = enrol_df.join(
    states_df.select("state", "district").distinct(),
    on=["state", "district"],
    how="left_anti"
).select(
    "state",
    "district"
).distinct() \
 .withColumn("is_invalid_location", F.lit(True))


# Create a pincode-based reference lookup containing the trusted
# state and district values.
#
# dropDuplicates() prevents duplicate reference records from
# multiplying rows when the lookup is joined to the enrolment data.
pincode_lookup_df = states_df.select(
    "pincode",
    F.col("state").alias("correct_state"),
    F.col("district").alias("correct_district")
).dropDuplicates(["pincode"])


# Attach the invalid-location flag and the reference location
# associated with each pincode.
updated_df = enrol_df.join(
    invalid_location_df,
    on=["state", "district"],
    how="left"
).join(
    pincode_lookup_df,
    on="pincode",
    how="left"
)


# Replace the state and district only when the original
# state-district combination is invalid and a valid pincode
# reference value is available.
enrol_df = updated_df.withColumn(
    "district",
    F.when(
        F.col("is_invalid_location").isNotNull() &
        F.col("correct_district").isNotNull(),
        F.col("correct_district")
    ).otherwise(F.col("district"))
).withColumn(
    "state",
    F.when(
        F.col("is_invalid_location").isNotNull() &
        F.col("correct_state").isNotNull(),
        F.col("correct_state")
    ).otherwise(F.col("state"))
).drop(
    "is_invalid_location",
    "correct_state",
    "correct_district"
)


StatementMeta(, 6dcd164d-fc23-45b0-b81a-95af71e120fd, 17, Finished, Available, Finished, False)

In [16]:
district_mapping = {
    "andhra_pradesh": {
        "khammam": {"telangana": "khammam"},
        "nalgonda": {"telangana": "nalgonda"},
        "sri_potti_sriramulu_nellore": {"andhra_pradesh": "spsr_nellore"},
        "nellore": {"andhra_pradesh": "spsr_nellore"},
        "hyderabad": {"telangana": "hyderabad"},
        "k_v_rangareddy": {"telangana": "ranga_reddy"},
        "rangareddi": {"telangana": "ranga_reddy"},
        "mahabubnagar": {"telangana" : "mahabubnagar"},
        "mahbubnagar": {"telangana" : "mahabubnagar"},
        "visakhapatnam" : {"andhra_pradesh":"visakhapatanam"}
    },
    "bihar": {
        "west_champaran": {"bihar": "pashchim_champaran"},
        "east_champaran": {"bihar": "purbi_champaran"},
        "purba_champaran": {"bihar": "purbi_champaran"},
        "purnea" : {"bihar": "purnia"},
        "samstipur" :{"bihar":"samastipur"},
        "monghyr" : {"bihar" : "munger"},
        "aurangabad(bh)" : {"bihar" : "aurangabad"}
    },
    "chhattisgarh": {
        "dakshin_bastar_dantewada": {"chhattisgarh": "dantewada"},
    },
    "gujarat": {
        "ahmedabad": {"gujarat": "ahmadabad"},
        "panchmahals": {"gujarat": "panch_mahals"},
        "surendra_nagar": {"gujarat": "surendranagar"},
    },
    "haryana": {
        "yamuna_nagar": {"haryana": "yamunanagar"},
    },
    "jammu_and_kashmir": {
        "punch": {"jammu_and_kashmir": "poonch"},
        "baramula": {"jammu_and_kashmir": "baramulla"},
        "kargil": {"ladakh": "kargil"},
        "leh": {"ladakh": "leh_ladakh"}
    },
    "jharkhand": {
        "purbi_singhbhum": {"jharkhand": "east_singhbum"},
        "pashchimi_singhbhum": {"jharkhand": "west_singhbhum"},
        "east_singhbhum": {"jharkhand": "east_singhbum"},
        "seraikela_kharsawan":{"jharkhand" : "saraikela_kharsawan"}
    },
    "karnataka": {
        "chickmagalur": {"karnataka": "chikkamagaluru"},
        "hasan": {"karnataka": "hassan"},
        "bijapur": {"karnataka": "vijayapura"},
        "shimoga": {"karnataka": "shivamogga"},
        "mysore": {"karnataka": "mysuru"},
        "belgaum": {"karnataka": "belagavi"},
        "tumkur": {"karnataka": "tumakuru"},
        "ramanagar": {"karnataka": "ramanagara"},
        "chikmagalur" : {"karnataka":"chikkamagaluru"},
        "bangalore_rural" : {"karnataka":"bengaluru_rural"}
    },
    "ladakh": {
        "leh": {"ladakh": "leh_ladakh"},
    },
    "madhya_pradesh": {
        "narsimhapur": {"madhya_pradesh": "narsinghpur"},
        "ashok_nagar" : {"madhya_pradesh":"ashoknagar"}
    },
    "maharashtra": {
        "ahmadnagar": {"maharashtra": "ahmednagar"},
        "mumbai(_sub_urban_)": {"maharashtra": "mumbai_suburban"},
        "mumbai(_sub_urban_)": {"maharashtra": "mumbai_suburban"},
        "mumbai_city": {"maharashtra": "mumbai"},
        "ahilyanagar" : {"maharashtra":"ahmednagar"},
        "ahmed_nagar" : {"maharashtra" : "ahmednagar"}
    },
    "mizoram": {
        "mammit": {"mizoram": "mamit"},
    },
    "odisha": {
        "angul": {"odisha": "anugul"},
        "subarnapur": {"odisha": "sonepur"},
        "baleswar": {"odisha": "baleshwar"},
        "balasore" : {"odisha": "baleshwar"},
        "jagatsinghpur" : {"odisha":"jagatsinghapur"}
    },
    "puducherry": {
        "puducherry": {"puducherry": "pondicherry"},
    },
    "punjab": {
        "sas_nagar_(mohali)": {"punjab": "s_a_s_nagar"},
        "sas_nagar" : {"punjab": "s_a_s_nagar"},
        "firozpur" : {"punjab":"firozepur"},
        "shaheed_bhagat_singh_nagar" :{"punjab" : "shahid_bhagat_singh_nagar"}
    },
    "rajasthan": {
        "jhunjhunun": {"rajasthan": "jhunjhunu"},
        "chittaurgarh": {"rajasthan": "chittorgarh"},
        "didwana_kuchaman": {"maharashtra":"nagpur"},
        "khairthal_tijara":{"rajasthan" : "alwar"}
    },
    "sikkim": {
        "east_sikkim": {"sikkim": "east_district"},
        "east": {"sikkim": "east_district"},
        "sikkim": {"sikkim": "east_district"},
        "north_sikkim": {"sikkim": "north_district"},
        "north": {"sikkim": "north_district"},
        "gangtok" : {"sikkim" : "east_district"},
        "mangan" : {"sikkim" : "north_district"}
    },
    "tamil_nadu": {
        "kancheepuram": {"tamil_nadu": "kanchipuram"},
        "kanyakumari": {"tamil_nadu": "kanniyakumari"},
        "tiruvallur": {"tamil_nadu": "thiruvallur"},
        "thoothukkudi" : {"tamil_nadu":"tuticorin"}
    },
    "telangana": {
        "k_v_rangareddy": {"telangana": "ranga_reddy"},
        "rangareddy": {"telangana": "ranga_reddy"},
    },
    "uttar_pradesh": {
        "allahabad": {"uttar_pradesh": "prayagraj"},
        "bara_banki": {"uttar_pradesh": "barabanki"},
        "bulandshahar": {"uttar_pradesh": "bulandshahr"},
        "sant_kabir_nagar": {"uttar_pradesh": "sant_kabeer_nagar"},
        "faizabad": {"uttar_pradesh": "ayodhya"},
        "sant_ravidas_nagar" : {"uttar_pradesh":"bhadohi"},
        "bagpat" : {"uttar_pradesh":"baghpat"}
    },
    "uttarakhand": {
        "udham_singh_nagar": {"uttarakhand": "udam_singh_nagar"},
        "garhwal" : {"uttarakhand" : "tehri_garhwal"}
    },
    "west_bengal": {
        "dakshin_dinajpur" : {"west_bengal": "dinajpur_dakshin"},
        "paschim_medinipur": {"west_bengal": "medinipur_west"},
        "north_dinajpur": {"west_bengal": "dinajpur_uttar"},
        "bardhaman": {"west_bengal": "purba_bardhaman"},
        "malda": {"west_bengal": "maldah"},
        "north_24_parganas": {"west_bengal": "24_paraganas_north"},
        "south_twenty_four_parganas": {"west_bengal": "24_paraganas_south"},
        "puruliya": {"west_bengal": "purulia"},
        "cooch_behar": {"west_bengal": "coochbehar"},
        "purba_medinipur": {"west_bengal": "medinipur_east"},
        "koch_bihar": {"west_bengal": "coochbehar"},
        "haora": {"west_bengal": "howrah"},
        "dakshin_dinajpur": {"west_bengal": "dinajpur_dakshin"},
        "barddhaman": {"west_bengal": "purba_bardhaman"},
        "south_24_parganas": {"west_bengal": "24_paraganas_south"},
        "uttar_dinajpur": {"west_bengal": "dinajpur_uttar"},
        "south_dinajpur": {"west_bengal": "dinajpur_dakshin"},
        "medinipur": {"west_bengal": "medinipur_west"},
        "north_twenty_four_parganas": {"west_bengal": "24_paraganas_south"},
        "east_midnapore": {"west_bengal": "medinipur_east"},
        "darjiling": {"west_bengal": "darjeeling"},
        "west_midnapore": {"west_bengal": "medinipur_west"},
        "hugli": {"west_bengal": "hooghly"},
        
    },
}



from pyspark.sql import functions as F

# Build mapping:
# "source_state\tsource_district" -> "target_state\ttarget_district"
mapping_expr = F.create_map(
    *[
        F.lit(item)
        for state, districts in district_mapping.items()
        for district, target in districts.items()
        for target_state, target_district in target.items()
        for item in (
            f"{state}\t{district}",
            f"{target_state}\t{target_district}"
        )
    ]
)

# Create source state-district key and map it to the target state-district
enrol_df = (
    enrol_df
    .withColumn(
        "_state_district_key",
        F.concat_ws("\t", F.col("state"), F.col("district"))
    )
    .withColumn(
        "_mapped_state_district",
        mapping_expr[F.col("_state_district_key")]
    )
    .withColumn(
        "state",
        F.coalesce(
            F.split(F.col("_mapped_state_district"), "\t").getItem(0),
            F.col("state")
        )
    )
    .withColumn(
        "district",
        F.coalesce(
            F.split(F.col("_mapped_state_district"), "\t").getItem(1),
            F.col("district")
        )
    )
    .drop("_state_district_key", "_mapped_state_district")
)

# Re-check state-district combinations against the trusted reference dataset.
missing_districts = (
    enrol_df
    .join(
        states_df.select("state", "district").distinct(),
        on=["state", "district"],
        how="left_anti"
    )
    .select("state", "district")
    .distinct()
)

display(missing_districts)


StatementMeta(, 6dcd164d-fc23-45b0-b81a-95af71e120fd, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 77576676-dead-4f65-bf12-4a08fd0ad65c)

In [17]:
bengaluru_df = enrol_df.filter(enrol_df["district"] == "bengaluru") \
              .select("state", "district", "pincode") \
              .distinct()

# Show the results
bengaluru_df.show()

StatementMeta(, 6dcd164d-fc23-45b0-b81a-95af71e120fd, 19, Finished, Available, Finished, False)

+---------+---------+-------+
|    state| district|pincode|
+---------+---------+-------+
|karnataka|bengaluru| 560039|
|karnataka|bengaluru| 560019|
|karnataka|bengaluru| 560069|
|karnataka|bengaluru| 560014|
|karnataka|bengaluru| 560046|
|karnataka|bengaluru| 560028|
|karnataka|bengaluru| 560052|
+---------+---------+-------+



In [18]:
# map bengaluru district
enrol_df = enrol_df.withColumn(
    "district",
    when(substring(col("pincode").cast("string"), 1, 3) == "560", "bengaluru_urban")
    .when(substring(col("pincode").cast("string"), 1, 3).isin("561","571", "562"), "bengaluru_rural")
    .otherwise(col("district"))
)

StatementMeta(, 6dcd164d-fc23-45b0-b81a-95af71e120fd, 20, Finished, Available, Finished, False)

In [19]:
missing_states = enrol_df.join(
    states_df, 
    on="state", 
    how="left_anti"
).select("state").distinct()

missing_states.show()


# Re-check state-district combinations against the trusted reference dataset.
missing_districts = (
    enrol_df
    .join(
        states_df.select("state", "district").distinct(),
        on=["state", "district"],
        how="left_anti"
    )
    .select("state", "district")
    .distinct()
)

missing_districts.show()


StatementMeta(, 6dcd164d-fc23-45b0-b81a-95af71e120fd, 21, Finished, Available, Finished, False)

+-----+
|state|
+-----+
+-----+

+-----+--------+
|state|district|
+-----+--------+
+-----+--------+



In [20]:
# Write using the single table name (if DSMLLH2 is your attached default)
enrol_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_enrolment_cleaned")

StatementMeta(, 6dcd164d-fc23-45b0-b81a-95af71e120fd, 22, Finished, Available, Finished, False)